# MuJoCo prep

## Importers

In [ ]:
# ---- Core stdlib ----
import os, re, json, math, sys, time, logging
from collections import defaultdict, deque
from pathlib import Path

# ---- Third-party ----
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
import warnings

# NEURON
from neuron import h  # keep this single, canonical import

# neuprint (API) — single alias
import neuprint as npt
from neuprint import (
    Client as NPClient,
    NeuronCriteria,            # you can use the class directly
    fetch_neurons,
    fetch_simple_connections,
    fetch_synapses,
    fetch_synapse_connections,
)
from neuprint.skeleton import (
    fetch_skeleton, heal_skeleton, upsample_skeleton,
    reorient_skeleton, attach_synapses_to_skeleton, skeleton_segments
)

# navis + its neuprint interface — different alias so it never shadows neuprint
import navis
import navis.interfaces.neuprint as navnp  # use navnp.* when you really need the navis bridge

# Optional UI utils
from IPython.display import display, clear_output
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib
matplotlib.use("Agg")   # do this once, near the top of the notebook

# ---- Configure neuprint client (hardcoded token, no alias collisions) ----
NEUPRINT_HOST = "https://neuprint.janelia.org"
NEUPRINT_DATASET = "manc:v1.2.1"
NEUPRINT_TOKEN = (
    "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9."
    "eyJlbWFpbCI6ImpwYWJsbzE5OTUwOUBnbWFpbC5jb20iLCJsZXZlbCI6Im5vYXV0aCIs"
    "ImltYWdlLXVybCI6Imh0dHBzOi8vbGgzLmdvb2dsZXVzZXJjb250ZW50LmNvbS9hL0FD"
    "ZzhvY0pCM3JUMm9HV19va1hBV2VMcVZoSDhoUVRXM1FoNUNnXzVFRW00Tm9VaWE0bS01"
    "QT1zOTYtYz9zej01MD9zej01MCIsImV4cCI6MTkyMjMxMjQ0Nn0."
    "Hmou-N87hM-qp1rVZinuQTKEwDymqioIM5LzX3s5bPk"
)

# Primary neuprint client (official API)
client = NPClient(NEUPRINT_HOST, dataset=NEUPRINT_DATASET, token=NEUPRINT_TOKEN)
npt.set_default_client(client)  # <- keep this on the neuprint module alias

# Navis→neuprint bridge client (separate module/alias)
navis_client = navnp.Client(NEUPRINT_HOST, dataset=NEUPRINT_DATASET, token=NEUPRINT_TOKEN)

# Convenience so your downstream code that expects `neu` to be the navis bridge keeps working
neu = navnp


import re


## Identify actuators

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
import re

def list_mjcf_actuators(xml_path):
    """
    Parse MJCF and return a DataFrame with actuator and joint metadata:
    [actuator_name, type, joint, gear, ctrlrange, side, limb, axis, direction_hint]
    """
    xml_path = Path(xml_path)
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = ""  # MuJoCo doesn't require XML namespaces

    # Collect joints for axis/direction hints if present
    joint_meta = {}
    for j in root.findall(".//joint"):
        name = j.attrib.get("name")
        axis = j.attrib.get("axis", "")
        # normalize axis label
        axis_hint = {"1 0 0":"x","0 1 0":"y","0 0 1":"z"}.get(axis.strip(), axis.strip())
        joint_meta[name] = {"axis_hint": axis_hint}

    rows = []
    # MuJoCo actuators are usually <motor>, <position>, <velocity>, <general>
    for tag in ("motor","position","velocity","general"):
        for a in root.findall(f".//{tag}"):
            aname = a.attrib.get("name", "")
            joint = a.attrib.get("joint", "")
            gear  = a.attrib.get("gear", "")
            ctrl  = a.attrib.get("ctrlrange", "")
            # crude, but practical: infer side/limb/joint from naming
            n = aname.lower()
            side = ("L" if re.search(r"\b(l|left)\b", n) else
                    "R" if re.search(r"\b(r|right)\b", n) else None)
            # common limb tokens seen in fly models
            limb = ( "fore"   if "fore"   in n or "front"  in n else
                     "mid"    if "mid"    in n else
                     "hind"   if "hind"   in n or "rear"   in n else None)
            # joint segment tokens (coxa/femur/tibia/tarsus) or body joints (head/wing/etc.)
            seg = None
            for t in ("coxa","femur","tibia","tarsus","trochanter","head","wing","abdomen","antenna"):
                if t in n:
                    seg = t; break

            axis_hint = joint_meta.get(joint,{}).get("axis_hint","")
            # sign hint (purely name-based; will be calibrated later)
            direction_hint = ("flex" if any(k in n for k in ["flex","retract","adduct","depress"])
                              else "extend" if any(k in n for k in ["extend","protract","abduct","elevate"])
                              else None)

            rows.append({
                "actuator_name": a.attrib.get("name",""),
                "type": tag,
                "joint": joint,
                "gear": gear,
                "ctrlrange": ctrl,
                "side": side,
                "limb_group": limb,
                "segment": seg,
                "axis_hint": axis_hint,
                "direction_hint": direction_hint
            })
    df = pd.DataFrame(rows).sort_values(["side","limb_group","segment","actuator_name"], na_position="last")
    return df

# EXAMPLE: point this to your model (adjust the path to the MJCF used by flybody)
df_act = list_mjcf_actuators("/path/to/local/flybody-main/flybody/fruitfly/build_fruitfly/fruitfly.xml")
print(df_act.to_string(index=False))


## Clean up MN naming for folders

In [ ]:
# === MN candidates → annotate (instance/type/origin/target) and rebucket by Instance ===
import os, json, re, shutil, math
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from neuprint import fetch_custom
except Exception as e:
    fetch_custom = None

MN_CAND_ROOT = Path("/path/to/local/digifly-legacy/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates")

def _sanitize_folder_name(s: str) -> str:
    """Filesystem-safe folder name (letters, digits, _, -, .)."""
    return re.sub(r'[^A-Za-z0-9._-]+', '-', str(s)).strip('-_.') or "unnamed"

def _discover_body_id_dirs(root: Path) -> list[int]:
    root = Path(root)
    if not root.exists():
        print(f"[mn-bucket] root missing: {root}")
        return []
    ids = []
    for p in sorted(root.iterdir()):
        if p.is_dir() and p.name.isdigit():
            ids.append(int(p.name))
    return ids

def _chunked(iterable, n):
    it = list(iterable)
    for i in range(0, len(it), n):
        yield it[i:i+n]

def _fetch_labels_for_ids(client, ids: list[int]) -> pd.DataFrame:
    """
    Batch fetch of (instance, type, origin/Origin, target/Target) for bodyIds.
    Returns DataFrame with columns: bodyId, instance, type, origin, target
    """
    if fetch_custom is None:
        raise RuntimeError("neuprint Python client is not available in this session.")
    if not ids:
        return pd.DataFrame(columns=["bodyId","instance","type","origin","target"])

    rows = []
    # Keep IN-list sizes reasonable (neuPrint is fine with a few hundred–1k per query)
    for chunk in _chunked(sorted(set(int(i) for i in ids)), 800):
        idlist = ",".join(str(int(i)) for i in chunk)
        cypher = f"""
        MATCH (n:Neuron)
        WHERE n.bodyId IN [{idlist}]
        RETURN n.bodyId AS bodyId,
               n.instance AS instance,
               n.type AS type,
               n.Origin AS Origin,  n.origin AS origin,
               n.Target AS Target,  n.target AS target
        """
        res = fetch_custom(cypher, client=client)
        df = res[0] if isinstance(res, tuple) else res
        if df is None or df.empty:
            continue
        rows.append(df)

    if not rows:
        return pd.DataFrame(columns=["bodyId","instance","type","origin","target"])

    df = pd.concat(rows, ignore_index=True)
    # coalesce Origin/origin → origin ; Target/target → target
    if "origin" not in df.columns: df["origin"] = np.nan
    if "Origin" in df.columns:
        df["origin"] = df["origin"].fillna(df["Origin"])
    if "target" not in df.columns: df["target"] = np.nan
    if "Target" in df.columns:
        df["target"] = df["target"].fillna(df["Target"])

    # standardize empties
    for c in ("instance","type","origin","target"):
        if c in df.columns:
            df[c] = df[c].replace({"": np.nan, "None": np.nan, "none": np.nan})

    keep_cols = ["bodyId","instance","type","origin","target"]
    for c in keep_cols:
        if c not in df.columns:
            df[c] = np.nan

    # drop dups (one row per bodyId)
    df = (df[keep_cols].drop_duplicates(subset=["bodyId"])
          .sort_values("bodyId")
          .reset_index(drop=True))
    return df

def annotate_and_rebucket_mn_candidates(client,
                                        root_dir: str | Path = MN_CAND_ROOT,
                                        dry_run=False,
                                        keep_old_structure_symlink=False,
                                        write_index_csv=True):
    """
    1) For each <root>/<ID>, query neuPrint to get instance/type/origin/target.
       Save to <root>/<ID>/metadata.json
    2) Move <root>/<ID> -> <root>/<Instance>/<ID>
       (Instance folder uses sanitized name; if missing, falls back to type, else 'UNLABELED').

    Args:
      client: neuprint.Client already configured to your dataset.
      root_dir: path to MN/MN_candidates
      dry_run: if True, only print planned changes.
      keep_old_structure_symlink: if True, leave a symlink at <root>/<ID> pointing to new location.
      write_index_csv: if True, also writes <root>/_mn_instance_index.csv
    """
    root = Path(root_dir).expanduser().resolve()
    ids = _discover_body_id_dirs(root)
    if not ids:
        print(f"[mn-bucket] No ID folders under: {root}")
        return pd.DataFrame()

    print(f"[mn-bucket] Found {len(ids)} ID folders.")

    # Batch pull labels
    meta = _fetch_labels_for_ids(client, ids)

    # per-ID save & rebucket plan
    plan = []
    for bid in ids:
        row = meta.loc[meta["bodyId"] == bid].head(1)
        inst = str(row["instance"].iloc[0]).strip() if not row.empty and pd.notna(row["instance"].iloc[0]) else ""
        typ  = str(row["type"].iloc[0]).strip() if not row.empty and pd.notna(row["type"].iloc[0]) else ""
        org  = str(row["origin"].iloc[0]).strip() if not row.empty and pd.notna(row["origin"].iloc[0]) else ""
        tgt  = str(row["target"].iloc[0]).strip() if not row.empty and pd.notna(row["target"].iloc[0]) else ""

        # destination folder name preference: instance → type → UNLABELED
        label_for_dir = inst or typ or "UNLABELED"
        dst_dir_name  = _sanitize_folder_name(label_for_dir)

        src = root / str(bid)
        dst = root / dst_dir_name / str(bid)

        plan.append({
            "bodyId": bid,
            "instance": inst,
            "type": typ,
            "origin": org,
            "target": tgt,
            "src": str(src),
            "dst": str(dst),
            "dst_type_dir": str(root / dst_dir_name)
        })

    # write metadata.json in each ID folder
    for item in plan:
        meta_path = Path(item["src"]) / "metadata.json"
        payload = {
            "bodyId": item["bodyId"],
            "instance": item["instance"],
            "type": item["type"],
            "origin": item["origin"],
            "target": item["target"]
        }
        if dry_run:
            print(f"[dry] write {meta_path}  ←  {payload}")
        else:
            try:
                with open(meta_path, "w") as f:
                    json.dump(payload, f, indent=2)
            except Exception as e:
                print(f"[warn] could not write {meta_path}: {e}")

    # show planned moves
    print("\n[mn-bucket] Planned moves:")
    for item in plan:
        print(f"  {item['src']}  →  {item['dst']}")

    if dry_run:
        print("[mn-bucket] Dry run only — no folders moved.")
    else:
        # perform moves
        for item in plan:
            src = Path(item["src"])
            dst = Path(item["dst"])
            try:
                dst.parent.mkdir(parents=True, exist_ok=True)
                # If a destination exists already, try to merge (move contents) then remove source
                if dst.exists():
                    # copy files inside src into dst (without clobbering existing files)
                    for child in src.iterdir():
                        target = dst / child.name
                        if target.exists():
                            continue
                        if child.is_dir():
                            shutil.move(str(child), str(target))
                        else:
                            shutil.move(str(child), str(dst))
                    # remove empty src
                    try:
                        src.rmdir()
                    except OSError:
                        pass
                else:
                    shutil.move(str(src), str(dst))

                # optional back-compat symlink at old location
                if keep_old_structure_symlink:
                    try:
                        link = src
                        if not link.exists():
                            link.symlink_to(dst)
                    except Exception as e:
                        print(f"[warn] symlink failed {src} → {dst}: {e}")

            except Exception as e:
                print(f"[error] move failed {src} → {dst}: {e}")

    # index CSV
    out_df = pd.DataFrame(plan).sort_values(["dst_type_dir","bodyId"])
    if write_index_csv:
        idx_path = root / "_mn_instance_index.csv"
        try:
            out_df.to_csv(idx_path, index=False)
            print(f"\n[mn-bucket] Wrote index → {idx_path}")
        except Exception as e:
            print(f"[warn] could not write index CSV: {e}")

    print("[mn-bucket] Done.")
    return out_df


In [ ]:
# # After you’ve set up `client = neuprint.Client(..., dataset=..., token=...)`
# annotate_and_rebucket_mn_candidates(client,
#     root_dir="/path/to/local/digifly-legacy/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates",
#     dry_run=False,                      # set True first to preview
#     keep_old_structure_symlink=False,   # set True if you want legacy links left behind
#     write_index_csv=True
# )


### Other Version OLD

In [ ]:
# import os, re, csv, shutil, json
# from pathlib import Path
# import pandas as pd

# # ---------- NeuPrint helpers ----------
# def get_full_type_from_neuprint(body_id, client):
#     """
#     Returns the *full* MN label from NeuPrint, prioritizing `instance`
#     (e.g., "MNwm34_PDMNp_L"). Falls back to `type`, `systematicType`, etc.
#     """
#     from neuprint import NeuronCriteria, fetch_neurons
#     df, _ = fetch_neurons(NeuronCriteria(bodyId=int(body_id)), client=client)
#     if df is None or df.empty:
#         return None

#     # Prefer instance first (human-readable + encodes side/nerve)
#     priority = ("instance", "type", "systematicType", "name", "cellType")
#     for col in priority:
#         if col in df.columns:
#             val = df.iloc[0].get(col, None)
#             if pd.notna(val):
#                 sval = str(val).strip()
#                 if sval:
#                     return sval
#     return None


# # ---------- Full-name parsing ----------
# # Nerve → coarse segment hints
# NERVE_BASES = {
#     "CvN": "head",  # cervical nerve → head/neck/antenna class
#     "DProN": "T1",  "VProN": "T1",  "ProAN": "T1",  "ProLN": "T1",
#     "ADMN": "T2",   "PDMN": "T2",   "MesoAN": "T2", "MesoLN": "T2",
#     "DMetaN": "T3", "MetaLN": "T3",
#     "AbN1": "Ab", "AbN2": "Ab", "AbN3": "Ab", "AbN4": "Ab", "AbNT": "Ab",
# }

# # Limb group routing by nerve
# NERVE_GROUP = {
#     "ProLN": "leg", "MesoLN": "leg", "MetaLN": "leg",
#     "ADMN": "wing", "PDMN": "wing",
#     "CvN": "head", "MesoAN": "wing_accessory", "ProAN": "pro_accessory",
#     "DProN": "pro_dorsal", "VProN": "pro_ventral", "DMetaN": "meta_dorsal",
#     "AbN1": "abdomen", "AbN2": "abdomen", "AbN3": "abdomen", "AbN4": "abdomen", "AbNT": "abdomen",
# }

# # Regexes for parsing
# SIDE_RX   = re.compile(r'(?:^|[_\-])(L|R|ND)(?:$|[_\-])', re.I)
# NERVE_RX  = re.compile(r'\b(CvN|DProN|VProN|ProAN|ProLN|ADMN|PDMN[pd]?|MesoAN|MesoLN|DMetaN|MetaLN|AbN[1-4]|AbNT)\b', re.I)
# THORAX_RX = re.compile(r'\b(T1|T2|T3)\b', re.I)

# def parse_full_mn_name(full_type):
#     """
#     Parse strings like "MNwm34_PDMNp_L" into components.
#     Returns dict(type_short, instance, nerve_token, nerve_base, side, thorax, segment_hint, group)

#     - side: 'L' | 'R' | 'ND' | ''  (raw token)
#     - thorax: 'T1'|'T2'|'T3' if inferred; else ''
#     - segment_hint: same as thorax but may carry 'Ab' for abdomen
#     """
#     instance = str(full_type).strip()
#     tokens = instance.split('_') if instance else []

#     # side
#     side = ""
#     if tokens and tokens[-1] in ("L", "R", "ND"):
#         side = tokens[-1]
#     else:
#         m = SIDE_RX.search(instance)
#         if m:
#             side = m.group(1).upper()

#     # nerve token & base (strip trailing branch letters like 'p'/'d')
#     nerve_token = ""
#     m = NERVE_RX.search(instance)
#     if m:
#         nerve_token = m.group(1)
#     elif len(tokens) >= 2:
#         # heuristic: middle token often nerve; take last non-side token
#         nerve_token = tokens[-2] if (tokens[-1] in ("L","R","ND") and len(tokens) >= 2) else tokens[-1]

#     nerve_base = re.sub(r'[a-z]+$', '', nerve_token) if nerve_token else ""

#     # thorax/segment hint
#     thorax = ""
#     m_t = THORAX_RX.search(instance)
#     if m_t:
#         thorax = m_t.group(1).upper()
#     else:
#         hint = NERVE_BASES.get(nerve_base, "")
#         thorax = hint if hint in ("T1","T2","T3") else ""

#     segment_hint = thorax if thorax else NERVE_BASES.get(nerve_base, "")

#     group = NERVE_GROUP.get(nerve_base, "")
#     type_short = tokens[0] if tokens else instance

#     return dict(
#         type_short=type_short,
#         instance=instance,
#         nerve_token=nerve_token,
#         nerve_base=nerve_base,
#         side=side,
#         thorax=thorax,
#         segment_hint=segment_hint,
#         group=group
#     )


# # ---------- Filesystem ops ----------
# def sanitize_folder_name(s):
#     """Make a filesystem-safe folder name (alnum, _, -, .)."""
#     return re.sub(r'[^A-Za-z0-9._-]+', '-', str(s)).strip('-_.') or "unnamed"

# def rename_mn_dirs(mn_root, client, dry_run=True, make_symlink=True, write_index_json=True):
#     """
#     Walk /MN/<short_type>/<id> and rename <short_type> → <instance> from NeuPrint.
#     Keeps <id> folder as-is. Optionally leaves a symlink from old to new and writes an index JSON.
#     """
#     mn_root = Path(mn_root).expanduser().resolve()
#     todo = []
#     index = []  # for optional JSON index
#     for type_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
#         short_type = type_dir.name
#         for id_dir in sorted(p for p in type_dir.iterdir() if p.is_dir() and p.name.isdigit()):
#             bid = int(id_dir.name)
#             full = get_full_type_from_neuprint(bid, client)
#             if not full:
#                 continue
#             dst_type = sanitize_folder_name(full)
#             if dst_type == short_type:
#                 continue
#             src = type_dir / str(bid)
#             dst = mn_root / dst_type / str(bid)
#             todo.append((src, dst, short_type, dst_type, bid, full))
#             index.append({
#                 "bodyId": bid,
#                 "old_type": short_type,
#                 "new_instance": full,
#                 "new_type_folder": dst_type,
#                 "old_path": str(src),
#                 "new_path": str(dst)
#             })

#     if not todo:
#         print("[rename] Nothing to rename.")
#         return []

#     print(f"[rename] Planned renames: {len(todo)}")
#     for src, dst, sshort, sfull_dir, bid, full in todo:
#         print(f"  {sshort}/{bid}  →  {sfull_dir}/{bid}")

#     if dry_run:
#         print("[rename] Dry run only. Re-run with dry_run=False to apply.")
#         return todo

#     for src, dst, sshort, sfull_dir, bid, full in todo:
#         dst.parent.mkdir(parents=True, exist_ok=True)
#         shutil.move(str(src), str(dst))

#         # remove empty old type dir
#         old_type_dir = src.parent
#         try:
#             if not any(old_type_dir.iterdir()):
#                 old_type_dir.rmdir()
#         except Exception:
#             pass

#         # optional back-compat symlink
#         if make_symlink:
#             try:
#                 old_type_dir.mkdir(parents=True, exist_ok=True)
#                 link = old_type_dir / str(bid)
#                 if not link.exists():
#                     link.symlink_to(dst)
#             except Exception as e:
#                 print(f"  [warn] symlink failed for {src} → {dst}: {e}")

#     if write_index_json:
#         idx_path = mn_root / "_rename_index.json"
#         try:
#             with open(idx_path, "w") as f:
#                 json.dump(index, f, indent=2)
#             print(f"[rename] Wrote index → {idx_path}")
#         except Exception as e:
#             print(f"[rename] Could not write index JSON: {e}")

#     print("[rename] Done.")
#     return todo


# # ---------- Mapping CSV from renamed dirs ----------
# # Toggle: for leg MNs, auto-fill a COARSE default joint/dof so you can run right away.
# COARSE_LEG_DEFAULTS = False   # set True to default to tibia/main, sign=+1

# def infer_joint_defaults_from_group(thorax, side_token, group):
#     """
#     For non-leg groups we can map to named actuators directly.
#     For legs we either leave joint/dof blank or give a coarse default.
#     Returns list of dict rows (may be empty).
#     """
#     side = {"L": "left", "R": "right"}.get(side_token, "")

#     if group == "wing":
#         if side == "left":  return [{"joint": "wing", "dof": "pitch", "actuator_name": "wing_pitch_left"}]
#         if side == "right": return [{"joint": "wing", "dof": "pitch", "actuator_name": "wing_pitch_right"}]
#         return []

#     if group == "abdomen":
#         return [{"joint": "abdomen", "dof": "main", "actuator_name": "abdomen"}]

#     if group == "head":
#         return [{"joint": "head", "dof": "main", "actuator_name": "head"}]

#     if group.endswith("accessory") or group.endswith("dorsal") or group.endswith("ventral"):
#         # ambiguous routes (leave blank for manual assignment)
#         return []

#     if group == "leg":
#         if not COARSE_LEG_DEFAULTS:
#             return []
#         # coarse default: tibia main on segment thorax/side
#         return [{"joint": "tibia", "dof": "main", "actuator_name": ""}]

#     return []





### Identify folders to rename

In [ ]:
# # MN_ROOT = "/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
# # rename_mn_dirs(MN_ROOT, client, dry_run=True)


# # ===== Run it with your paths =====
# MN_ROOT = "/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
# _ = build_mapping_from_dirs_with_targets(MN_ROOT, client, out_csv="mn_to_actuator_mapping.csv")

### Run Renaming

In [ ]:
# rename_mn_dirs(MN_ROOT, client, dry_run=False, make_symlink=True)

### Audit folder names

In [ ]:
# from pathlib import Path
# import pandas as pd

# def audit_mn_dirs(mn_root, client):
#     from neuprint import NeuronCriteria, fetch_neurons
#     mn_root = Path(mn_root).expanduser().resolve()
#     rows = []
#     for type_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
#         short = type_dir.name
#         for id_dir in sorted(p for p in type_dir.iterdir() if p.is_dir()):
#             bid = id_dir.name
#             if not bid.isdigit():
#                 rows.append(dict(id=bid, short=short, neu_full=None, action="skip: non-numeric id"))
#                 continue
#             df,_ = fetch_neurons(NeuronCriteria(bodyId=int(bid)), client=client)
#             if df is None or df.empty:
#                 rows.append(dict(id=bid, short=short, neu_full=None, action="skip: neuprint empty"))
#                 continue
#             full = None
#             for col in ("type","systematicType","instance","name","cellType"):
#                 if col in df.columns and pd.notna(df.iloc[0].get(col, None)):
#                     full = str(df.iloc[0][col]).strip(); 
#                     if full: break
#             if not full:
#                 rows.append(dict(id=bid, short=short, neu_full=None, action="skip: no full name"))
#             elif full == short:
#                 rows.append(dict(id=bid, short=short, neu_full=full, action="skip: same name"))
#             else:
#                 rows.append(dict(id=bid, short=short, neu_full=full, action="RENAME"))
#     audit_df = pd.DataFrame(rows, columns=["id","short","neu_full","action"])
#     print(audit_df.to_string(index=False))
#     return audit_df

# # usage
# MN_ROOT = "/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
# _ = audit_mn_dirs(MN_ROOT, client)


## Build Muscle to Actuator control CSV (Up-to-Date, run all 4)

In [ ]:
# from pathlib import Path
# import json
# import pandas as pd

# def build_mapping_from_dirs(
#     mn_root,
#     out_csv="mn_to_actuator_mapping.csv",
#     prefer_metadata=True,
#     client=None,             # optional: only used if you want to refetch missing metadata
#     refetch_if_missing=False # set True to try neuPrint when metadata.json is missing/incomplete
# ):
#     """
#     Creates (or updates) a CSV with rows for each MN found under mn_root.
#     Assumes ID folders have been rebucketed under instance folders:
#         <mn_root>/<Instance>/<ID>
#     Reads metadata from <ID>/metadata.json (instance/type/origin/target) when present.

#     New columns added: 'origin', 'target'
#     """

#     mn_root = Path(mn_root).expanduser().resolve()
#     rows = []

#     # --- optional helper if you want to refetch missing metadata on the fly ---
#     def _maybe_refetch(bid: int):
#         if (client is None) or (not refetch_if_missing):
#             return {}
#         try:
#             # lightweight batched query replaced with one-body fallback
#             from neuprint import fetch_custom
#             cy = f"""
#             MATCH (n:Neuron) WHERE n.bodyId = {int(bid)}
#             RETURN n.instance AS instance, n.type AS type,
#                    coalesce(n.origin, n.Origin) AS origin,
#                    coalesce(n.target, n.Target) AS target
#             """
#             res = fetch_custom(cy, client=client)
#             df = res[0] if isinstance(res, tuple) else res
#             if df is None or df.empty:
#                 return {}
#             r = df.iloc[0].to_dict()
#             # normalize empties
#             for k in ("instance","type","origin","target"):
#                 v = r.get(k, None)
#                 if v is None: continue
#                 s = str(v).strip()
#                 r[k] = s if s else None
#             return r
#         except Exception:
#             return {}

#     for instance_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
#         instance_name = instance_dir.name  # expected sanitized INSTANCE (e.g., MNfl27_ProLN_L)

#         # derive thorax/side/group/etc from the instance folder name
#         info = parse_full_mn_name(instance_name)
#         thorax       = info.get("thorax","") or ""
#         side_token   = info.get("side","")
#         side         = {"L": "left", "R": "right"}.get(side_token, "")
#         group        = info.get("group","")
#         nerve_base   = info.get("nerve_base","")
#         segment_hint = info.get("segment_hint","")

#         for id_dir in sorted(p for p in instance_dir.iterdir() if p.is_dir() and p.name.isdigit()):
#             bid = int(id_dir.name)

#             # 1) Try metadata.json inside the ID folder
#             meta_path = id_dir / "metadata.json"
#             meta = {}
#             if prefer_metadata and meta_path.exists():
#                 try:
#                     with open(meta_path, "r") as f:
#                         meta = json.load(f) or {}
#                 except Exception:
#                     meta = {}

#             # normalize metadata fields; fall back to instance folder name
#             mn_instance = (str(meta.get("instance") or instance_name)).strip()
#             mn_type     = (str(meta.get("type") or "")).strip() or mn_instance  # keep your old behavior
#             origin      = (str(meta.get("origin") or "")).strip()
#             target      = (str(meta.get("target") or "")).strip()

#             # 2) If still missing and refetch is enabled, hit neuPrint
#             if refetch_if_missing and (not origin or not target or not mn_instance):
#                 ref = _maybe_refetch(bid)
#                 mn_instance = ref.get("instance", mn_instance)
#                 mn_type     = ref.get("type", mn_type)
#                 origin      = ref.get("origin", origin)
#                 target      = ref.get("target", target)

#             # 3) Build default rows as before (can be multiple rows via your helper)
#             defaults = infer_joint_defaults_from_group(thorax, side_token, group)

#             if not defaults:
#                 rows.append({
#                     "mn_type": mn_instance,        # same meaning as before (full instance)
#                     "mn_id": bid,
#                     "thorax": thorax,
#                     "side": side,
#                     "nerve_base": nerve_base,
#                     "segment_hint": segment_hint,
#                     "group": group,
#                     # NEW:
#                     "origin": origin,
#                     "target": target,
#                     # editable mapping fields:
#                     "joint": "",
#                     "dof": "",
#                     "action": "",
#                     "gain": 1.0,
#                     "bias": 0.0,
#                     "sign": "",
#                     "actuator_name": ""
#                 })
#             else:
#                 for d in defaults:
#                     rows.append({
#                         "mn_type": mn_instance,
#                         "mn_id": bid,
#                         "thorax": thorax,
#                         "side": side,
#                         "nerve_base": nerve_base,
#                         "segment_hint": segment_hint,
#                         "group": group,
#                         # NEW:
#                         "origin": origin,
#                         "target": target,
#                         # pre-filled mapping suggestions:
#                         "joint": d.get("joint", ""),
#                         "dof": d.get("dof", ""),
#                         "action": "",
#                         "gain": 1.0,
#                         "bias": 0.0,
#                         "sign": "",
#                         "actuator_name": d.get("actuator_name", "")
#                     })

#     cols = [
#         "mn_type", "mn_id", "thorax", "side",
#         "nerve_base", "segment_hint", "group",
#         # NEW columns (right after your anatomical hints):
#         "origin", "target",
#         # editable mapping fields (unchanged):
#         "joint", "dof", "action", "gain", "bias", "sign", "actuator_name"
#     ]
#     df = pd.DataFrame(rows, columns=cols)

#     # stable sort: by mn_type then mn_id
#     if not df.empty:
#         df = df.sort_values(["mn_type", "mn_id"]).reset_index(drop=True)

#     df.to_csv(out_csv, index=False)
#     print(f"[mapping] Wrote {len(df)} rows → {out_csv}")
#     print("  Fill any blanks (esp. leg joint/dof), set sign (+1/-1). Now you also have origin/target from neuPrint.")
#     return df


In [ ]:
# build_mapping_from_dirs(
#     "/path/to/local/digifly-legacy/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates",
#     out_csv="/path/to/local/digifly-legacy/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates/mn_to_actuator_mapping.csv",
#     client=client,
#     refetch_if_missing=True   # only if you want to query neuPrint for gaps
# )



In [ ]:
# import re
# import pandas as pd
# import numpy as np

# # ---- SAFE string helpers ----
# def s(x):
#     """Return '' for None/NaN, else str(x)."""
#     if x is None: return ""
#     if isinstance(x, float) and (np.isnan(x)): return ""
#     return str(x)

# def slower(x):
#     v = s(x)
#     return v.lower()

# # ---- extractors ----
# def _seg_num(thorax_str: str) -> str:
#     m = re.search(r'T([123])', s(thorax_str), flags=re.I)
#     return m.group(1) if m else ""

# def _side_lr(side_str: str) -> str:
#     ss = slower(side_str)
#     if ss in ("l","left"):  return "left"
#     if ss in ("r","right"): return "right"
#     return ""

# def _contains(txt, *keys):
#     t = slower(txt)
#     return all(k.lower() in t for k in keys)

# def _any(txt, *keys):
#     t = slower(txt)
#     return any(k.lower() in t for k in keys)

# # Optional: if you have parse_full_mn_name() already defined, we can use it
# def _fallback_from_mn_type(mn_type):
#     # expects parse_full_mn_name to be available in your session
#     try:
#         info = parse_full_mn_name(s(mn_type))
#         thorax = info.get("thorax","") or info.get("segment_hint","")
#         side_token = info.get("side","")
#         side = {"L":"left","R":"right"}.get(side_token, "")
#         group = info.get("group","")
#         return thorax, side, group
#     except Exception:
#         return "", "", ""

# # ---- main mapper from Target text → actuator_name (+joint/dof hints) ----
# def map_target_to_actuator(thorax, side, target, limb_group=None):
#     seg = _seg_num(thorax)
#     lr  = _side_lr(side)
#     t   = slower(target)
    
#     # TTM (tergotrochanteral muscle) → use femur actuator as trochanter proxy
#     if _any(t, "ttm", "tergotrochanter", "tergo-trochanter"):
#         if not seg or not lr:
#             return "", "", ""
#         return "femur", "", f"femur_T{seg}_{lr}"

#     # Antenna / head / wing / abdomen first (don’t require seg/lr)
#     if _any(t, "antenna", "antennal"):
#         if _any(t, "abduct", "abductor"):
#             return "antenna", "abduct", f"antenna_abduct_{lr or 'left'}"
#         if _any(t, "twist", "rotat"):
#             return "antenna", "twist", f"antenna_twist_{lr or 'left'}"
#         return "antenna", "", f"antenna_{lr or 'left'}"

#     if _any(t, "wing"):
#         return "wing", "pitch", f"wing_pitch_{lr or 'left'}"

#     if _any(t, "abdomen", "abdominal"):
#         if _any(t, "abduct"):
#             return "abdomen", "abduct", "abdomen_abduct"
#         return "abdomen", "", "abdomen"

#     if _any(t, "head"):
#         if _any(t, "abduct"):
#             return "head", "abduct", "head_abduct"
#         if _any(t, "twist", "yaw"):
#             return "head", "twist", "head_twist"
#         return "head", "", "head"

#     # Legs (need segment + side)
#     lg = slower(limb_group)
#     if lg == "leg" or _any(t, "tibia", "femur", "coxa", "trochanter", "tarsus"):
#         if not seg or not lr:
#             return "", "", ""  # cannot choose a T{1/2/3}_{left/right} actuator

#         # Tarsus
#         if _any(t, "tarsus2", "tarsal2"):
#             return "tarsus", "", f"tarsus2_T{seg}_{lr}"
#         if _any(t, "tarsus", "tarsal"):
#             return "tarsus", "", f"tarsus_T{seg}_{lr}"

#         # Tibia
#         if _any(t, "tibia"):
#             return "tibia", "", f"tibia_T{seg}_{lr}"

#         # Coxa
#         if _any(t, "coxa"):
#             if _any(t, "abduct", "abductor"):
#                 return "coxa", "abduct", f"coxa_abduct_T{seg}_{lr}"
#             if _any(t, "twist", "rotat"):
#                 return "coxa", "twist", f"coxa_twist_T{seg}_{lr}"
#             return "coxa", "", f"coxa_T{seg}_{lr}"

#         # Trochanter (incl. accessory trochanter) → femur as safe proxy
#         if _any(t, "trochanter", "acc. tr", "accessory tr", "acc.tr", "tr "):
#             return "femur", "", f"femur_T{seg}_{lr}"

#         # Femur
#         if _any(t, "femur", "femoral"):
#             if _any(t, "twist", "rotat"):
#                 return "femur", "twist", f"femur_twist_T{seg}_{lr}"
#             return "femur", "", f"femur_T{seg}_{lr}"

#     # Unknown
#     return "", "", ""

# # ----- hard overrides (by instance or by bodyId) -----
# MN_OVERRIDES_BY_TYPE = {
#     # “wm” is a misnomer; these are T2 leg jump MNs
#     "MNwm34_PDMNp_L": {"group": "leg", "thorax": "T2", "side": "left",
#                        "joint": "femur", "dof": "", "actuator_name": "femur_T2_left"},
#     "MNwm34_PDMNp_R": {"group": "leg", "thorax": "T2", "side": "right",
#                        "joint": "femur", "dof": "", "actuator_name": "femur_T2_right"},
# }
# MN_OVERRIDES_BY_ID = {
#     # if you prefer to key by bodyId instead, add entries like:
#     # 10068: {"group":"leg","thorax":"T2","side":"left","joint":"femur","dof":"","actuator_name":"femur_T2_left"},
#     # 10110: {"group":"leg","thorax":"T2","side":"right","joint":"femur","dof":"","actuator_name":"femur_T2_right"},
# }


# def autofill_actuators(csv_path, out_csv=None, target_col="target"):
#     """
#     Fills joint/dof/actuator_name in your mapping CSV using:
#       1) hard overrides (MN_OVERRIDES_BY_TYPE / _BY_ID)
#       2) map_target_to_actuator() heuristics (incl. TTM rule)
#     Only writes when we can determine an actuator_name.
#     """
#     import pandas as pd
#     from pathlib import Path
#     def s(x): return "" if (x is None or (isinstance(x, float) and pd.isna(x))) else str(x)

#     csv_path = Path(csv_path).expanduser().resolve()
#     if out_csv is None:
#         out_csv = csv_path
#     else:
#         out_csv = Path(out_csv).expanduser().resolve()

#     df = pd.read_csv(csv_path)

#     # make sure columns exist
#     for c in ["mn_type","mn_id","thorax","side","group","joint","dof","actuator_name"]:
#         if c not in df.columns:
#             df[c] = ""

#     filled = 0
#     for i, row in df.iterrows():
#         # ----- per-MN overrides -----
#         mn_type = s(row.get("mn_type",""))
#         mn_id   = row.get("mn_id", None)
#         try:
#             # normalize possible floats to int
#             if isinstance(mn_id, float) and not pd.isna(mn_id):
#                 mn_id = int(mn_id)
#         except Exception:
#             pass

#         # by instance
#         if mn_type in MN_OVERRIDES_BY_TYPE:
#             o = MN_OVERRIDES_BY_TYPE[mn_type]
#             # update context fields (thorax/side/group) if provided
#             for k in ("group","thorax","side"):
#                 if k in o and o[k]:
#                     df.at[i, k] = o[k]
#             df.at[i, "joint"]         = o.get("joint", s(row.get("joint","")))
#             df.at[i, "dof"]           = o.get("dof", s(row.get("dof","")))
#             df.at[i, "actuator_name"] = o.get("actuator_name", s(row.get("actuator_name","")))
#             filled += 1
#             continue

#         # by bodyId
#         if mn_id in MN_OVERRIDES_BY_ID:
#             o = MN_OVERRIDES_BY_ID[mn_id]
#             for k in ("group","thorax","side"):
#                 if k in o and o[k]:
#                     df.at[i, k] = o[k]
#             df.at[i, "joint"]         = o.get("joint", s(row.get("joint","")))
#             df.at[i, "dof"]           = o.get("dof", s(row.get("dof","")))
#             df.at[i, "actuator_name"] = o.get("actuator_name", s(row.get("actuator_name","")))
#             filled += 1
#             continue

#         # ----- heuristic mapping -----
#         tgt = s(row.get(target_col, ""))
#         joint, dof, name = map_target_to_actuator(
#             s(row.get("thorax","")),
#             s(row.get("side","")),
#             tgt,
#             s(row.get("group",""))
#         )
#         if name:
#             df.at[i, "joint"]         = joint
#             df.at[i, "dof"]           = dof
#             df.at[i, "actuator_name"] = name
#             filled += 1

#     df.to_csv(out_csv, index=False)
#     print(f"[autofill] Updated {filled} rows → {out_csv}")
#     return df



In [ ]:
# autofill_actuators(
#     "/path/to/local/digifly-legacy/Python Code/Phase 1 - Neuprint Data Imports/export_swc/MN/MN_candidates/mn_to_actuator_mapping.csv"
# )

## MN to MuJoCo full CSV Mapping

### Step 1: Fetch  labels for bodyIds

In [ ]:
import re, os, json, shutil
from pathlib import Path
import pandas as pd

def fetch_mn_labels(body_id, client):
    """
    Returns a dict with ALL the columns your dataset has that are useful for labeling,
    using only columns that exist in your schema.
    """
    from neuprint import fetch_neurons, NeuronCriteria
    df, _ = fetch_neurons(NeuronCriteria(bodyId=int(body_id)), client=client)
    if df is None or df.empty:
        return None

    row = df.iloc[0]
    # Pull only if present in columns
    def get(c, default=None):
        return row[c] if (c in df.columns and pd.notna(row[c])) else default

    D = {
        "bodyId":        int(body_id),
        "instance":      str(get("instance", "") or "").strip(),
        "type":          str(get("type", "") or "").strip(),
        "class":         str(get("class", "") or "").strip(),
        "subclass":      str(get("subclass", "") or "").strip(),
        "subclassabbr":  str(get("subclassabbr", "") or "").strip(),
        "group":         str(get("group", "") or "").strip(),
        "hemilineage":   str(get("hemilineage", "") or "").strip(),
        "exitNerve":     str(get("exitNerve", "") or "").strip(),
        "entryNerve":    str(get("entryNerve", "") or "").strip(),
        "somaNeuromere": str(get("somaNeuromere", "") or "").strip(),  # e.g. T1/T2/T3
        "somaSide":      str(get("somaSide", "") or "").strip(),        # e.g. left/right
        "vfbId":         str(get("vfbId", "") or "").strip(),
        "status":        str(get("status","") or "").strip(),
        "statusLabel":   str(get("statusLabel","") or "").strip(),
        "description":   str(get("description","") or "").strip(),
        "predictedNt":   str(get("predictedNt","") or "").strip(),
    }

    # If side is missing, infer from instance suffix _L/_R if present
    if not D["somaSide"]:
        m = re.search(r"_(L|R)(?:$|_)", D["instance"])
        if m:
            D["somaSide"] = {"L":"left","R":"right"}[m.group(1)]

    # Normalize a few convenience flags
    inst = D["instance"]
    D["is_motor"] = (D["class"].lower() == "wm" or "motor" in D["group"].lower())
    D["is_leg"]   = ("leg" in D["subclass"].lower()) or ("LegNp" in D["subclass"]) or ("ProLN" in D["exitNerve"] or "MesoLN" in D["exitNerve"] or "MetaLN" in D["exitNerve"])
    D["is_wing"]  = ("wing" in D["group"].lower()) or ("ADMN" in D["exitNerve"] or "PDMN" in D["exitNerve"])
    return D
    
# ----- smarter grouping from neuprint labels -----

# small, explicit exception table for stubborn mislabels
# keys can be an instance (preferred) or a short type
SMART_OVERRIDES = {
    r"^MNwm\d+": {"group": "leg"},  # <- GOOD
}


def _apply_overrides(text, default_group):
    for pat, fix in SMART_OVERRIDES.items():
        if re.search(pat, text or "", flags=re.IGNORECASE):
            return fix.get("group", default_group)
    return default_group

def smarter_group_from_labels(labels: dict) -> tuple[str, str]:
    """
    Return (group, thorax) using neuprint fields with sensible priority:
      1) subclass / systematicType / instance tokens (LegNp(T), Wing, etc.)
      2) class/group fallbacks
      3) exitNerve LAST (unreliable for MNwm34/TTMn cases)
    """
    inst = (labels.get("instance") or labels.get("type") or "").strip()
    syst = (labels.get("systematicType") or "").strip()
    subc = (labels.get("subclass") or "").strip()
    cls  = (labels.get("class") or "").strip()
    grp  = (labels.get("group") or "").strip()
    exitNerve = (labels.get("exitNerve") or "").strip()

    text = " | ".join([inst, syst, subc, cls, grp, exitNerve])

    # 1) explicit tokens that trump nerves
    # Leg neuropil
    m = re.search(r"LegNp\(T([123])\)", text, flags=re.IGNORECASE)
    if m:
        thorax = f"T{m.group(1)}"
        group  = "leg"
        return _apply_overrides(inst, group), thorax

    if re.search(r"\bLegNp(T?[123])?\b", text, flags=re.IGNORECASE):
        # if no digit, fall back to somaNeuromere if present
        thorax = labels.get("somaNeuromere", "")
        thorax = thorax if thorax in ("T1","T2","T3") else ""
        group  = "leg"
        return _apply_overrides(inst, group), thorax

    # Wing tokens
    if re.search(r"\bWing(Np|)|\bTTM\b|\bTTMn\b", text, flags=re.IGNORECASE):
        # T2 by default for wing motors unless neuromere says otherwise
        thorax = labels.get("somaNeuromere", "T2")
        thorax = thorax if thorax in ("T1","T2","T3") else "T2"
        group  = "wing"
        return _apply_overrides(inst, group), thorax

    # Antenna/head
    if re.search(r"\bantenna\b|\bCvN\b", text, flags=re.IGNORECASE):
        return "head", ""

    # Abdomen tokens
    if re.search(r"\bAbN[1-4]\b|\bAbNT\b|\babdomen\b", text, flags=re.IGNORECASE):
        return "abdomen", ""

    # 2) use neuromere for thorax when we already know it’s a motor MN
    if re.search(r"\bmotor\b", text, flags=re.IGNORECASE):
        nm = labels.get("somaNeuromere", "")
        if nm in ("T1","T2","T3"):
            # fall back group based on nerve if we can
            if "Leg" in subc or "Leg" in syst:
                return "leg", nm
            if "Wing" in subc or "Wing" in syst:
                return "wing", nm

    # 3) last-resort: exit nerve hints (weak!)
    if re.search(r"\b(ProLN|MesoLN|MetaLN)\b", exitNerve):
        thorax = dict(ProLN="T1", MesoLN="T2", MetaLN="T3")[re.search(r"(ProLN|MesoLN|MetaLN)", exitNerve).group(1)]
        return "leg", thorax
    if re.search(r"\b(ADMN|PDMN)\b", exitNerve):
        # many wing MNs, but NOT ALWAYS (e.g., MNwm34)
        nm = labels.get("somaNeuromere", "T2")
        thorax = nm if nm in ("T1","T2","T3") else "T2"
        group = _apply_overrides(inst, "wing")
        return group, thorax

    # If nothing matched, return neutral
    return _apply_overrides(inst, ""), labels.get("somaNeuromere", "") if labels.get("somaNeuromere","") in ("T1","T2","T3") else ""


### Step 2: Rename MN type directories to the instance label and write a JSON sidecar

In [ ]:
def sanitize_folder_name(s):
    return re.sub(r'[^A-Za-z0-9._-]+', '-', str(s))

def ensure_label_json(dir_for_id: Path, labels: dict):
    sidecar = dir_for_id / "labels.json"
    with sidecar.open("w") as f:
        json.dump(labels, f, indent=2)
    return sidecar

def rename_mn_dirs_to_instance(mn_root, client, dry_run=True, make_symlink=True):
    """
    Walks MN/<type_like>/<bodyId> and renames <type_like> → <instance>
    (e.g., MNwm34 → MNwm34_PDMNp_L), leaving the bodyId subfolder intact.
    Also writes MN/<instance>/<bodyId>/labels.json with neuPrint labels.
    """
    from pathlib import Path
    import shutil

    mn_root = Path(mn_root).expanduser().resolve()
    todo = []

    for type_dir in sorted(p for p in mn_root.iterdir() if p.is_dir()):
        # FIX: use 'p' consistently in the generator below (not 'id_dir')
        for id_dir in sorted(p for p in type_dir.iterdir() if p.is_dir() and p.name.isdigit()):
            bid = int(id_dir.name)
            lab = fetch_mn_labels(bid, client)
            if not lab:
                continue
            inst = lab["instance"] or lab["type"] or type_dir.name
            dst_type = sanitize_folder_name(inst)

            # If already named correctly, still ensure labels.json exists
            if dst_type == type_dir.name:
                ensure_label_json(id_dir, lab)
                continue

            src = id_dir
            dst = mn_root / dst_type / id_dir.name
            todo.append((src, dst, type_dir.name, dst_type, lab))

    if not todo:
        print("[rename] Nothing to rename.")
        return

    print(f"[rename] Planned renames: {len(todo)}")
    for src, dst, old, new, _ in todo:
        print(f"  {old}/{src.name}  →  {new}/{dst.name}")

    if dry_run:
        print("[rename] Dry run only. Re-run with dry_run=False to apply.")
        return

    for src, dst, old, new, lab in todo:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        ensure_label_json(dst, lab)

        # remove old type dir if empty
        old_dir = src.parent
        try:
            if not any(old_dir.iterdir()):
                old_dir.rmdir()
        except Exception:
            pass

        # optional symlink from old path to new
        if make_symlink:
            try:
                old_dir.mkdir(parents=True, exist_ok=True)
                link = old_dir / dst.name
                if not link.exists():
                    link.symlink_to(dst)
            except Exception as e:
                print(f"  [warn] symlink failed: {e}")

    print("[rename] Done.")


### Step 3: Smarter CSV enrichment

In [ ]:
# simple mapping from (group, side) → actuator names you showed earlier
WING_ACT = {"left":"wing_pitch_left", "right":"wing_pitch_right"}
HEAD_ACT = "head"
AB_ACT   = "abdomen"

def enrich_mapping_csv(csv_path, client, coarse_leg_defaults=False):
    df = pd.read_csv(csv_path)

    new_rows = []
    for _, r in df.iterrows():
        bid = int(r["mn_id"])
        lab = fetch_mn_labels(bid, client)  # your existing helper: returns dict with instance/type/subclass/exitNerve/...
        if not lab:
            new_rows.append(r)
            continue

        # recompute group/thorax smartly
        group, thorax = smarter_group_from_labels(lab)
        if group:
            r["group"] = group   # optional: keep this column
        if thorax:
            r["thorax"] = thorax

        # fill side from instance suffix if empty
        if not str(r.get("side","")).strip():
            side = "left" if re.search(r"[_-]L\b", (lab.get("instance") or ""), re.IGNORECASE) else \
                   ("right" if re.search(r"[_-]R\b", (lab.get("instance") or ""), re.IGNORECASE) else "")
            r["side"] = side

        # smarter default actuator guesses (only if actuator_name is blank)
        if not str(r.get("actuator_name","")).strip():
            if group == "wing":
                r["joint"], r["dof"], r["actuator_name"] = ("wing","pitch", "wing_pitch_left" if r["side"]=="left" else ("wing_pitch_right" if r["side"]=="right" else ""))
            elif group == "abdomen":
                r["joint"], r["dof"], r["actuator_name"] = ("abdomen","main","abdomen")
            elif group == "head":
                r["joint"], r["dof"], r["actuator_name"] = ("head","main","head")
            elif group == "leg" and coarse_leg_defaults:
                # give you something runnable; you can refine later
                r["joint"], r["dof"] = ("tibia","main")
                if thorax in ("T1","T2","T3") and r["side"] in ("left","right"):
                    r["actuator_name"] = f"tibia_{thorax}_{r['side']}"
                # leave sign blank for manual tuning
        new_rows.append(r)

    out = pd.DataFrame(new_rows, columns=df.columns)
    out.to_csv(csv_path, index=False)
    print(f"[enrich] updated → {csv_path}")


In [ ]:
import json, re
import pandas as pd

# ROI → coarse body region + thoracic segment + side
_ROI_PATTERNS = [
    # Legs
    (re.compile(r"LegNp\(T(?P<thorax>[123])\)\((?P<side>L|R)\)"), "leg"),
    # Wings (dorsal mesothoracic nerves)
    (re.compile(r"PDMN\((?P<side>L|R)\)"), "wing"),   # Posterior Dorsal Meso Nerve
    (re.compile(r"ADMN\((?P<side>L|R)\)"), "wing"),   # Anterior Dorsal Meso Nerve
    # Head / antenna via Cervical nerve
    (re.compile(r"CvN\((?P<side>L|R)\)"), "head"),
    # Abdomen nerves (AbN1..4, AbNT)
    (re.compile(r"AbN(?P<abnum>[1-4])\((?P<side>L|R)\)"), "abdomen"),
    (re.compile(r"AbNT"), "abdomen"),
]

def _first_match_roi(roi_keys):
    """
    Given a set of ROI keys (strings), return inferred (group, thorax, side).
    """
    group = thorax = side = ""
    for key in roi_keys:
        for pat, g in _ROI_PATTERNS:
            m = pat.search(key)
            if not m:
                continue
            group = g
            if "thorax" in m.groupdict():
                thorax = f"T{m.group('thorax')}"
            if "side" in m.groupdict():
                side = {"L": "left", "R": "right"}[m.group("side")]
            # once we have a decisive match, stop
            if group in ("leg", "wing", "head", "abdomen"):
                return group, thorax, side
    return group, thorax, side  # possibly empty strings

def _parse_roiinfo_field(val):
    """
    neuPrint 'roiInfo' can be dict-like or a JSON string; we just need its keys.
    Returns a set of ROI names present for this neuron.
    """
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return set()
    if isinstance(val, dict):
        return set(val.keys())
    if isinstance(val, str):
        try:
            obj = json.loads(val)
            if isinstance(obj, dict):
                return set(obj.keys())
        except Exception:
            # Sometimes roiInfo is a comma-separated blob; split as a fallback
            return set(re.findall(r"[A-Za-z0-9()\-+_]+", val))
    return set()

def _coarse_actuator_guess(group, thorax, side):
    """
    Optional guess to make rows immediately runnable.
    Legs remain blank (you’ll map to the exact joint/axis later).
    """
    if group == "wing":
        return "wing_pitch_left"  if side == "left"  else ("wing_pitch_right" if side == "right" else "")
    if group == "abdomen":
        return "abdomen"
    if group == "head":
        return "head"
    return ""  # leg, accessory, unknown → leave blank

def enrich_mapping_with_roi(csv_path, client, backup=True):
    """
    Enrich your mn_to_actuator_mapping.csv using neuPrint fields:
    - instance, type, group, subclass, exitNerve, somaSide
    - roiInfo (to fix leg/wing/head/abdomen + thorax + side)

    Updates columns: group, thorax, side, actuator_name (coarse guess for non-legs).
    """
    from neuprint import fetch_neurons, NeuronCriteria as NC

    df = pd.read_csv(csv_path)
    if backup:
        df.to_csv(csv_path + ".bak", index=False)

    rows_out = []
    filled = 0

    # Iterate row dicts so we can easily mutate and compare
    for row in df.to_dict(orient="records"):
        try:
            bid = int(row.get("mn_id"))
        except Exception:
            rows_out.append(row)
            continue

        # start from existing row; copy so we can compare
        new_row = dict(row)

        # Fetch neuPrint metadata
        meta, _ = fetch_neurons(NC(bodyId=bid), client=client)
        if meta is None or meta.empty:
            rows_out.append(new_row)
            continue

        meta0 = meta.iloc[0]

        # 1) ROI-driven inference
        roi_keys = _parse_roiinfo_field(meta0.get("roiInfo"))
        g_roi, t_roi, s_roi = _first_match_roi(roi_keys)

        # 2) Side fallback: somaSide if ROI didn’t give it
        if not s_roi:
            soma_side = str(meta0.get("somaSide") or "").strip().lower()
            if soma_side in ("left", "right"):
                s_roi = soma_side

        # 3) Group fallback: neuPrint 'group' or 'subclass'
        g_np = str(meta0.get("group") or "").strip().lower()
        if not g_roi:
            if "leg" in g_np:      g_roi = "leg"
            elif "wing" in g_np:   g_roi = "wing"
            elif "head" in g_np:   g_roi = "head"
            elif "abd" in g_np:    g_roi = "abdomen"

        # 4) Special-case known types (e.g., TTMn are leg MNs in T2 via PDMN)
        inst = str(meta0.get("instance") or "")
        typ  = str(meta0.get("type") or "")
        desc = str(meta0.get("description") or "")
        if ("TTMn" in typ) or ("TTMn" in inst) or ("TTMn" in desc):
            # TRUST ROI for side/thorax; if that failed, force leg/T2
            g_roi = g_roi or "leg"
            t_roi = t_roi or "T2"

        # 5) Apply to row (only overwrite if we have something)
        if g_roi: new_row["group"]  = g_roi
        if t_roi: new_row["thorax"] = t_roi
        if s_roi: new_row["side"]   = s_roi

        # 6) Coarse actuator guess for non-leg systems
        if not str(new_row.get("actuator_name","")).strip():
            guess = _coarse_actuator_guess(new_row.get("group",""), new_row.get("thorax",""), new_row.get("side",""))
            if guess:
                new_row["actuator_name"] = guess

        # --- Did anything change? Compare specific columns safely ---
        changed = False
        for col in ("group","thorax","side","actuator_name"):
            old_val = str(row.get(col, "") or "")
            new_val = str(new_row.get(col, "") or "")
            if new_val and (new_val != old_val):
                changed = True

        if changed:
            filled += 1
        rows_out.append(new_row)

    df_out = pd.DataFrame(rows_out, columns=df.columns)
    df_out.to_csv(csv_path, index=False)
    print(f"[roi-enrich] Updated {filled} rows using neuPrint roiInfo/metadata → {csv_path}")
    return df_out


### Step 4: Run mapping

In [ ]:
MN_ROOT = "/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN"
CSV_PATH = "/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_SWC/MN/mn_to_actuator_mapping.csv"

# 1) Rename type folders to full neuPrint `instance` and write labels.json in each ID folder
# rename_mn_dirs_to_instance(MN_ROOT, client, dry_run=False)

# 2) Enrich CSV with the correct thorax/side/group and actuator guesses
enrich_mapping_csv(CSV_PATH, client, coarse_leg_defaults=False)  # set True if you want tibia defaults

In [ ]:
# CSV_PATH = "/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_SWC/MN/mn_to_actuator_mapping.csv"
enrich_mapping_with_roi(CSV_PATH, client)  # makes a .bak and rewrites CSV in place

## Step 5: MuJoCo Output

In [ ]:
# ========= NEURON → MuJoCo (notebook-style env) : COMPACT + KNOBS =========
import os, sys, math
from pathlib import Path
import numpy as np, pandas as pd, imageio

# -------------------- PATHS --------------------
RESULT_DIR   = Path("/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A")
MAPPING_CSV  = Path("/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping.csv")
FLY_ASSETS   = Path("/path/to/local/flybody-main/flybody/fruitfly/assets")
WORLD_XML    = FLY_ASSETS / "floor.xml"      # includes fruitfly.xml
VIDEO_OUT    = RESULT_DIR / "mujoco_run_env.mp4"

# Optional (so `import neuro_adapter` etc. works)
NEURO_FLY_SRC = Path("/path/to/local/digifly-legacy/Python Code/Phase 3 - NEURON-to-MuJoCo Bridge/neuro_fly")
if str(NEURO_FLY_SRC) not in sys.path:
    sys.path.insert(0, str(NEURO_FLY_SRC))

os.environ.setdefault("MUJOCO_GL", "glfw")

# -------------------- TOP-LEVEL KNOBS --------------------
# Input → activation
USE_SPIKES        = True        # True: use spike_times.csv; False: detect crossings from voltages_somas.csv
V_THRESH_MV       = -20.0       # used if USE_SPIKES=False
TAU_RISE_MS       = 0.8         # spike→activation kernel (ms)
TAU_DECAY_MS      = 6.0
KERNEL_SCALE      = 1.0         # multiply kernel
POST_SMOOTH_WIN   = 0           # moving-average window (samples) after summing (0 = off)

# Time & visibility
EXTEND_TOTAL_MS   = 120.0       # extend signals so motion is visible
SLOWMO_FACTOR     = 60.0         # >1 = slower video (writes more frames)

# Control remap into MuJoCo ctrlrange
CTRL_GAIN         = 8.0         # amplifies dimensionless activation→control
CTRL_BIAS_TO_MID  = False        # add mid-bias so you see motion around neutral

# Camera (absolute zoom, non-compounding)
CAM_NAME          = "track2"    # "hero", "side", "back", "track1", ...
CAM_DIST_FACTOR   = 10.0        # distance = model.stat.extent * this factor
CAM_FOVY_DEG      = 75.0        # absolute FOV in degrees

# Actuator strength (T2 extensor chain both sides)
GEAR_MULT         = 2.5         # modest gear multiplier (too large → unstable)
FORCERANGE_MULT   = 10.0        # widen force caps so gear isn't just saturating
BOOST_TARGETS     = (
    "coxa_T2_left","femur_T2_left","tibia_T2_left",
    "coxa_T2_right","femur_T2_right","tibia_T2_right",
)

# Loss/traction knobs (reduce losses along the chain and improve ground grip)
LOSS_DAMPING_MULT = 0.45        # <1 reduces damping on selected DOFs
LOSS_ARMATURE_MULT= 0.5         # <1 reduces armature on selected DOFs
LEG_JOINTS        = ("coxa_T2_left","femur_T2_left","tibia_T2_left",
                     "coxa_T2_right","femur_T2_right","tibia_T2_right")
FLOOR_FRICTION    = (1.8, 0.005, 0.002)   # (slide, spin, roll) for floor geom

# Explosive push knobs (multiplicative burst on selected actuators)
ADD_EXPLOSIVE_BURST = True
BURST_T0_MS         = None       # None = auto (first nonzero); or a number
BURST_WINDOW_MS     = (0.0, 45.0)  # relative window [offset, offset+dur] from t0
BURST_GAIN          = 10.0        # 4..10 typical

# Optional debug pulse to guarantee a visible push
ADD_JUMP_PULSE    = False       # keep False if using ADD_EXPLOSIVE_BURST
PULSE_MS_START    = 0.0
PULSE_MS_DUR      = 120.0
PULSE_STRENGTH    = 0.9         # 0..1, pushes toward ctrl upper end during pulse window

# -------------------- MAP: TTMn → full T2 extensor chain --------------------
def build_ttmn_overrides(mn_left=10068, mn_right=10110):
    return {
        int(mn_left):  ["coxa_T2_left","femur_T2_left","tibia_T2_left"],
        int(mn_right): ["coxa_T2_right","femur_T2_right","tibia_T2_right"],
    }
MN_OVERRIDES = build_ttmn_overrides()

# -------------------- CSV mapping → {actuator:[sources]} --------------------
def load_mapping(csv_path, overrides=None):
    df = pd.read_csv(csv_path)
    if "actuator_name" not in df.columns: raise ValueError("mapping CSV must contain 'actuator_name'")
    df = df[df["actuator_name"].astype(str).str.strip().ne("")]
    for c, default in (("gain",1.0),("bias",0.0)):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(default) if c in df.columns else default
    df["sign"] = (df["sign"] if "sign" in df.columns else 1.0)
    def _sgn(x):
        try: return 1.0 if float(x)>=0 else -1.0
        except: return -1.0 if str(x).strip() in ("-1","-","minus") else 1.0
    df["sign"] = df["sign"].apply(_sgn)

    # apply overrides (replace those MNs, then add one row per target actuator)
    if overrides:
        try:
            ids = set(int(k) for k in overrides.keys())
            if "mn_id" in df.columns: df = df[~df["mn_id"].astype(int).isin(ids)].copy()
        except: pass
        add = []
        for bid, names in overrides.items():
            names = names if isinstance(names,(list,tuple)) else [names]
            for name in names:
                add.append({"mn_type":"","mn_id":int(bid),"actuator_name":str(name),"gain":1.0,"bias":0.0,"sign":1.0})
        if add: df = pd.concat([df, pd.DataFrame(add)], ignore_index=True)

    by_act = {}
    for _, r in df.iterrows():
        by_act.setdefault(str(r["actuator_name"]).strip(), []).append(
            dict(mn_id=int(r["mn_id"]), gain=float(r["gain"]), bias=float(r["bias"]), sign=float(r["sign"]))
        )
    return by_act

# -------------------- spikes/voltages → activation series --------------------
def spikes_to_activation(spike_times_ms, t_grid_ms, tau_rise=1.0, tau_decay=6.0, scale=1.0):
    a = np.zeros_like(t_grid_ms, float)
    if len(spike_times_ms)==0: return a
    dt = float(np.median(np.diff(t_grid_ms))); tmax = 6.0*tau_decay
    k_t = np.arange(0.0, tmax+dt, dt)
    k = (1.0 - np.exp(-k_t/tau_rise)) * np.exp(-k_t/tau_decay) * scale
    for ts in spike_times_ms:
        i = int(np.searchsorted(t_grid_ms, ts)); j = min(i+len(k), len(a))
        a[i:j] += k[:j-i]
    return a

def build_actuator_controls(result_dir: Path, mapping_by_act,
                            use_spikes=True, v_thresh=0.0,
                            tau_rise=1.0, tau_decay=6.0, scale=1.0,
                            post_smooth_win=0):
    spikes_csv, volts_csv = result_dir/"spike_times.csv", result_dir/"voltages_somas.csv"
    if not volts_csv.exists(): raise FileNotFoundError(f"Missing voltages_somas.csv in {result_dir}")
    vdf = pd.read_csv(volts_csv); t_ms = vdf["t_ms"].to_numpy(float)

    if use_spikes and spikes_csv.exists():
        sdf = pd.read_csv(spikes_csv)
        id2spk = {int(nid): grp["spike_time_ms"].to_numpy(float) for nid, grp in sdf.groupby("neuron_id")} if not sdf.empty else {}
    else:
        id2spk = {}
        for col in vdf.columns:
            if col=="t_ms": continue
            v = vdf[col].to_numpy(float)
            above = (v[:-1] < v_thresh) & (v[1:] >= v_thresh)
            id2spk[int(col)] = t_ms[1:][above]

    acts = {}
    for act, sources in mapping_by_act.items():
        acc = np.zeros_like(t_ms, float)
        for s in sources:
            a = spikes_to_activation(id2spk.get(int(s["mn_id"]), np.array([],float)), t_ms,
                                     tau_rise=tau_rise, tau_decay=tau_decay, scale=scale)
            acc += s["sign"]*s["gain"]*a
        acc += sum(s["bias"] for s in sources)
        if post_smooth_win and post_smooth_win>1:
            w = int(post_smooth_win); acc = np.convolve(acc, np.ones(w)/w, mode="same")
        acts[act] = acc
    return t_ms, acts

# -------------------- time stretch + ctrl remap --------------------
def extend_signals(t_ms, act_controls, total_ms=150.0):
    t0 = float(t_ms[0]); dt = float(np.median(np.diff(t_ms)))
    new_t = np.arange(t0, total_ms+1e-9, dt)
    if len(new_t) <= len(t_ms): return t_ms, act_controls
    out = {}
    for k, sig in act_controls.items():
        sig = np.asarray(sig, float)
        tail = np.full(len(new_t)-len(sig), (sig[-1] if len(sig) else 0.0), float)
        out[k] = np.concatenate([sig, tail])
    return new_t, out

def remap_to_ctrlrange(act_controls, env, gain=3.0, bias_to_mid=True):
    m = env.physics.model; out = {}
    name2id = {m.id2name(i,'actuator'): i for i in range(m.nu)}
    for nm, series in act_controls.items():
        aid = name2id.get(nm)
        if aid is None: out[nm] = np.asarray(series, float); continue
        lo, hi = m.actuator_ctrlrange[aid]; mid = 0.5*(lo+hi); rng = max(hi-mid, mid-lo, 1e-6)
        u = np.asarray(series, float) * float(gain)
        if bias_to_mid: u = mid + 0.5*rng*u
        out[nm] = np.clip(u, lo, hi) if lo<hi else u
    return out

# -------------------- env build (notebook-style) --------------------
def build_env_from_xml(world_xml: Path):
    try:
        from neuro_adapter.env_loader import load_env
        return load_env(xml_path=str(world_xml))
    except Exception: pass
    try:
        import flybody.fly_envs as fb
        if hasattr(fb,"xml_world"): return fb.xml_world(str(world_xml))
    except Exception: pass
    from dm_control import mujoco as dm_mujoco
    physics = dm_mujoco.Physics.from_xml_path(str(world_xml))
    class SimpleEnv:
        def __init__(self, physics): self.physics=physics
        def reset(self): self.physics.reset()
        def step(self,*_): self.physics.step()
        def control_timestep(self): return float(self.physics.timestep())
        class _Spec: 
            def __init__(self,lo,hi): self.minimum,self.maximum=lo,hi
        def action_spec(self):
            m=self.physics.model
            lo=m.actuator_ctrlrange[:,0] if m.actuator_ctrlrange.size else np.full((m.nu,),-1.0)
            hi=m.actuator_ctrlrange[:,1] if m.actuator_ctrlrange.size else np.full((m.nu,), 1.0)
            return SimpleEnv._Spec(lo,hi)
    return SimpleEnv(physics)

# -------------------- camera: absolute zoom (non-compounding) --------------------
def set_camera_zoom(env, cam_name="track2", distance_factor=8.0, fovy_deg=70.0):
    ph = env.physics; m = ph.model
    cam_id = next((i for i in range(m.ncam) if (m.id2name(i,'camera') or "")==cam_name), -1)
    if cam_id<0: print(f"[camera] '{cam_name}' not found"); return cam_name
    center = np.array(m.stat.center, float)
    v = m.cam_pos[cam_id].copy() - center
    if np.linalg.norm(v)<1e-9: v = np.array([-1.0,0.0,1.0], float)
    v /= (np.linalg.norm(v)+1e-12)
    dist = float(m.stat.extent)*float(distance_factor)
    m.cam_pos[cam_id] = center + v*dist
    m.cam_fovy[cam_id] = float(np.clip(fovy_deg, 10.0, 150.0))
    ph.forward()
    try: ph._reset_contexts()
    except AttributeError: pass
    print(f"[camera] '{cam_name}': pos={m.cam_pos[cam_id]}, fovy={m.cam_fovy[cam_id]:.1f}")
    return cam_name

# -------------------- actuator strength (cap first, then gear) --------------------
def boost_actuator_strength_on_env(env, targets, gear_mult=12.5, forcerange_mult=12.0):
    m = env.physics.model; name2id = {m.id2name(i,'actuator'): i for i in range(m.nu)}
    changed=[]
    for nm in targets:
        aid = name2id.get(nm)
        if aid is None: continue
        lo,hi = m.actuator_forcerange[aid]
        if lo<hi:
            span=(hi-lo)*float(forcerange_mult); mid=0.5*(hi+lo)
            m.actuator_forcerange[aid,0]=mid-0.5*span; m.actuator_forcerange[aid,1]=mid+0.5*span
        if m.actuator_gear.shape[1]>=1: m.actuator_gear[aid,0]*=float(gear_mult)
        changed.append(nm)
    env.physics.forward()
    try: env.physics._reset_contexts()
    except AttributeError: pass
    print("[boost] actuators:", changed)

# -------------------- losses & traction --------------------
def ease_leg_losses(env, joint_names, damping_mult=0.5, armature_mult=0.6):
    m = env.physics.model
    jname2id = {m.id2name(i,'joint'): i for i in range(m.njnt)}
    changed=[]
    for jn in joint_names:
        jid = jname2id.get(jn)
        if jid is None: continue
        dof = int(m.jnt_dofadr[jid])
        if 0 <= dof < m.dof_damping.size:
            m.dof_damping[dof] *= float(damping_mult)
        if 0 <= dof < m.dof_armature.size:
            m.dof_armature[dof] *= float(armature_mult)
        changed.append(jn)
    env.physics.forward()
    print("[loss] adjusted joints:", changed)

def improve_floor_grip(env, friction=(1.5, 0.005, 0.002)):
    m = env.physics.model; changed=False
    for i in range(m.ngeom):
        nm = (m.id2name(i,'geom') or "").lower()
        if 'floor' in nm or 'ground' in nm:
            m.geom_friction[i,:] = friction
            changed=True
    if changed: env.physics.forward()
    print(f"[floor] friction set to {friction}")

# -------------------- optional: explosive burst & debug pulse --------------------
def add_explosive_burst(t_ms, act_controls, targets, t0_ms=None,
                        burst_ms=(10.0, 45.0), burst_gain=6.0):
    if not targets: return act_controls
    t = np.asarray(t_ms, float)
    # auto start
    if t0_ms is None:
        starts=[]
        for nm in targets:
            u = np.asarray(act_controls.get(nm, np.zeros_like(t)))
            nz = np.where(np.abs(u) > (0.02*np.nanmax(np.abs(u))+1e-9))[0]
            if len(nz): starts.append(t[nz[0]])
        t0_ms = (min(starts) if starts else t[0])
    start = t0_ms + float(burst_ms[0])
    end   = t0_ms + float(burst_ms[1])
    w = (t >= start) & (t <= end)
    out = {k: np.array(v, float, copy=True) for k,v in act_controls.items()}
    for nm in targets:
        if nm in out: out[nm][w] *= float(burst_gain)
    return out

def add_jump_pulse(t_ms, act_controls, targets, start_ms=0.0, dur_ms=120.0, strength=0.9):
    from copy import deepcopy
    out = deepcopy(act_controls)
    i0 = int(np.searchsorted(t_ms, start_ms)); i1 = int(np.searchsorted(t_ms, start_ms+dur_ms))
    for nm in targets:
        if nm not in out: continue
        x = np.asarray(out[nm], float); lo, hi = np.nanmin(x), np.nanmax(x)
        x[i0:i1] = (1.0-strength)*lo + strength*hi
        out[nm] = x
    return out

# -------------------- render (env.physics loop, no camera tweaks here) --------------------
def render_env_with_controls(env, t_ms, act_controls, video_out,
                             cam_name="back", slowmo=0.0, H=1920, W=1280, render_hz=120):
    ph = env.physics; m = ph.model
    # resample controls to sim steps
    dt = float(ph.timestep()); dt_ms = 1000.0*dt
    T = int(math.ceil((t_ms[-1]-t_ms[0]) / dt_ms)) + 1
    times_sim = t_ms[0] + np.arange(T)*dt_ms
    name2id = {m.id2name(i,'actuator'): i for i in range(m.nu)}
    U = np.zeros((m.nu, T), float)
    for nm, series in act_controls.items():
        aid = name2id.get(nm)
        if aid is None: continue
        U[aid,:] = np.interp(times_sim, t_ms, np.asarray(series,float))
        lo,hi = m.actuator_ctrlrange[aid]
        if lo<hi: U[aid,:] = np.clip(U[aid,:], lo, hi)

    video_out.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(str(video_out), fps=render_hz)
    try: env.reset()
    except Exception: ph.reset()
    frame0 = ph.render(height=H, width=W, camera_id=cam_name)
    imageio.imwrite(str(Path(video_out).with_suffix(".preview.png")), frame0)

    base_steps_per_frame = max(1, int(round((1.0/render_hz)/dt)))
    steps_per_frame = max(1, int(round(base_steps_per_frame/max(1.0, slowmo))))
    frames=0
    for k in range(T):
        ph.data.ctrl[:] = U[:,k]; ph.step()
        if k % steps_per_frame == 0:
            writer.append_data(ph.render(height=H, width=W, camera_id=cam_name)); frames+=1
    writer.close()
    print(f"[video] {video_out}  frames={frames}  fps={render_hz}  slowmo×{slowmo}  (~{frames/render_hz:.1f}s)")

# ===================== RUN =====================
# 1) mapping + controls (choose spikes vs voltages)
mapping_by_act = load_mapping(MAPPING_CSV, overrides=MN_OVERRIDES)
t_ms, act_controls = build_actuator_controls(
    RESULT_DIR, mapping_by_act,
    use_spikes=USE_SPIKES, v_thresh=V_THRESH_MV,
    tau_rise=TAU_RISE_MS, tau_decay=TAU_DECAY_MS, scale=KERNEL_SCALE,
    post_smooth_win=POST_SMOOTH_WIN
)

# 2) extend window for visibility
t_long, act_controls_long = extend_signals(t_ms, act_controls, total_ms=EXTEND_TOTAL_MS)

# 3) env + traction/loss + boosts + camera
env = build_env_from_xml(WORLD_XML)
improve_floor_grip(env, friction=FLOOR_FRICTION)
ease_leg_losses(env, LEG_JOINTS, damping_mult=LOSS_DAMPING_MULT, armature_mult=LOSS_ARMATURE_MULT)
boost_actuator_strength_on_env(env, BOOST_TARGETS, gear_mult=GEAR_MULT, forcerange_mult=FORCERANGE_MULT)
CAM = set_camera_zoom(env, cam_name=CAM_NAME, distance_factor=CAM_DIST_FACTOR, fovy_deg=CAM_FOVY_DEG)

# 4) (optional) add explosive burst OR debug pulse on the *dimensionless* signals
act_controls_mod = dict(act_controls_long)
if ADD_EXPLOSIVE_BURST:
    act_controls_mod = add_explosive_burst(
        t_long, act_controls_mod, BOOST_TARGETS,
        t0_ms=BURST_T0_MS, burst_ms=BURST_WINDOW_MS, burst_gain=BURST_GAIN
    )
if ADD_JUMP_PULSE:
    act_controls_mod = add_jump_pulse(
        t_long, act_controls_mod, BOOST_TARGETS,
        start_ms=PULSE_MS_START, dur_ms=PULSE_MS_DUR, strength=PULSE_STRENGTH
    )

# 5) remap to ctrlrange (use env model so ranges are exact)
act_controls_scaled = remap_to_ctrlrange(act_controls_mod, env, gain=CTRL_GAIN, bias_to_mid=CTRL_BIAS_TO_MID)

# 6) render
render_env_with_controls(env, t_long, act_controls_scaled, VIDEO_OUT,
                         cam_name=CAM, slowmo=SLOWMO_FACTOR, H=1920, W=1280, render_hz=60)


## DIAGNOSTICS

In [ ]:
# ==== ONE-CELL, SELF-CONTAINED DIAGNOSTICS (defines all names it uses) ====
import os, math, numpy as np, pandas as pd

# --- EDIT THESE IF NEEDED ---
RESULT_DIR  = RESULT_DIR if 'RESULT_DIR' in globals() else \
    "/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/hemi_runs/hemi_09A"
MAPPING_CSV = MAPPING_CSV if 'MAPPING_CSV' in globals() else \
    "/path/to/local/digifly-legacy/Python Code/Phase 2 - Neuprint-to-NEURON/export_swc/MN/mn_to_actuator_mapping.csv"
MJCF_XML    = MJCF_XML if 'MJCF_XML' in globals() else \
    "/path/to/local/flybody-main/flybody/fruitfly/assets/fruitfly.xml"

# # AFTER: TTMn drive tibias instead of femurs
# MN_OVERRIDES = { 10068: "tibia_T2_left", 10110: "tibia_T2_right" }



# MN_OVERRIDES = {
#     10068: "femur_T2_left",
#     10110: "femur_T2_right",
# }

# ---- helpers (local, no external deps) ----
import pandas as pd, numpy as np

def load_mapping_strict(csv_path, overrides=None):
    df = pd.read_csv(csv_path)

    # Clean actuator_name
    df["actuator_name"] = df.get("actuator_name", "").astype(str).str.strip()
    bad = df["actuator_name"].isin(["", "nan", "None"])
    df = df[~bad].copy()

    # Clean numerics
    for c, default in (("gain", 1.0), ("bias", 0.0)):
        df[c] = pd.to_numeric(df.get(c, default), errors="coerce").fillna(default)

    def _sgn(x):
        try: return 1.0 if float(x) >= 0 else -1.0
        except: return -1.0 if str(x).strip() in ("-1","-","minus") else 1.0
    df["sign"] = df.get("sign", 1.0).apply(_sgn)

    # Optional overrides
    if overrides:
        for bid, name in overrides.items():
            msk = pd.to_numeric(df["mn_id"], errors="coerce").astype("Int64") == int(bid)
            if msk.any():
                df.loc[msk, "actuator_name"] = str(name).strip()
            else:
                df = pd.concat([df, pd.DataFrame([{
                    "mn_type":"", "mn_id":int(bid), "actuator_name":str(name).strip(),
                    "gain":1.0, "bias":0.0, "sign":1.0
                }])], ignore_index=True)

    by_act = {}
    for _, r in df.iterrows():
        nm = str(r["actuator_name"]).strip()
        by_act.setdefault(nm, []).append(
            dict(mn_id=int(r["mn_id"]),
                 gain=float(r["gain"]), bias=float(r["bias"]), sign=float(r["sign"]))
        )
    return by_act




def _spikes_to_activation(spike_times_ms, t_grid_ms, tau_rise=1.0, tau_decay=6.0, scale=1.0):
    a = np.zeros_like(t_grid_ms, dtype=float)
    if len(spike_times_ms) == 0: return a
    dt = float(np.median(np.diff(t_grid_ms)))
    tmax = 6.0*tau_decay
    k_t = np.arange(0.0, tmax+dt, dt)
    k = (1.0 - np.exp(-k_t/tau_rise)) * np.exp(-k_t/tau_decay) * scale
    for ts in spike_times_ms:
        i = int(np.searchsorted(t_grid_ms, ts))
        j = min(i + len(k), len(a))
        if i < len(a): a[i:j] += k[:j-i]
    return a

def _build_act_controls(result_dir, mapping_by_act, tau_rise=1.0, tau_decay=6.0,
                        scale=1.0, from_voltages=False, v_thresh=0.0):
    result_dir = os.fspath(result_dir)
    vdf = pd.read_csv(os.path.join(result_dir, "voltages_somas.csv"))
    t_ms = vdf["t_ms"].to_numpy(dtype=float)
    id2spk = {}
    spikes_csv = os.path.join(result_dir, "spike_times.csv")
    if (not from_voltages) and os.path.exists(spikes_csv):
        sdf = pd.read_csv(spikes_csv)
        if not sdf.empty:
            sdf["neuron_id"] = sdf["neuron_id"].astype(int)
            id2spk = {nid: grp["spike_time_ms"].to_numpy(dtype=float)
                      for nid, grp in sdf.groupby("neuron_id")}
    if not id2spk:  # fallback from voltages
        for col in vdf.columns:
            if col == "t_ms": continue
            v = vdf[col].to_numpy(dtype=float)
            above = (v[:-1] < v_thresh) & (v[1:] >= v_thresh)
            ts = t_ms[1:][above]
            try: nid = int(col)
            except: continue
            id2spk[nid] = ts

    act_controls = {}
    for act, sources in mapping_by_act.items():
        acc = np.zeros_like(t_ms, dtype=float)
        for s in sources:
            spikes = id2spk.get(int(s["mn_id"]), np.array([], float))
            a = _spikes_to_activation(spikes, t_ms, tau_rise, tau_decay, scale=scale)
            acc += s["sign"] * s["gain"] * a
        acc += sum(s["bias"] for s in sources)
        act_controls[act] = acc
    return t_ms, act_controls

def _load_model_acts(xml_path):
    import mujoco
    m = mujoco.MjModel.from_xml_path(str(xml_path))
    names = [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(m.nu)]
    ctrl = [tuple(m.actuator_ctrlrange[i]) for i in range(m.nu)]
    return names, ctrl, m

# ---- run diagnostics ----


model_acts, ctrl_ranges, _m = _load_model_acts(MJCF_XML)
# Rebuild mapping → controls
mapping_by_act = load_mapping_strict(MAPPING_CSV, overrides=MN_OVERRIDES)

# use your existing builder
t_ms, act_controls = _build_act_controls(RESULT_DIR, mapping_by_act,
                                             tau_rise=1.0, tau_decay=6.0, scale=1.0,
                                             from_voltages=False, v_thresh=0.0)

# Safe magnitudes print
print("— Actuator signal magnitudes —")
nonzero_acts = []
for nm, sig in act_controls.items():
    nm_str = str(nm)
    mx = float(np.nanmax(np.abs(sig))) if len(sig) else 0.0
    print(f"{nm_str:24s} max|u|={mx:.6g}")
    if mx > 1e-6:
        nonzero_acts.append(nm_str)
print("nonzero actuators:", nonzero_acts or "NONE")

print("nonzero actuators:", nonzero_acts or "NONE")

print("\n— Mapping ∩ Model —")
mapped = sorted(set(act_controls.keys()))
present = sorted(set(mapped) & set(model_acts))
missing = sorted(set(mapped) - set(model_acts))
print("present:", present)
print("missing:", missing)

# (Optional) show first 10 model actuators + ranges
print("\n— First 10 model actuators & ctrlranges —")
for i,(nm,rg) in enumerate(list(zip(model_acts, ctrl_ranges))[:10]):
    lo, hi = rg
    print(f"{i:2d} {nm:24s} ctrlrange=({lo:.3g}, {hi:.3g})")

# (Optional) inject a visible pulse on a known actuator to prove movement
try:
    import mujoco
    test_act = None
    for candidate in ["abdomen", "femur_T2_left", "femur_T1_left", "wing_pitch_left"]:
        if candidate in model_acts:
            test_act = candidate; break
    if test_act:
        dt = float(np.median(np.diff(t_ms)))
        extra_ms = 1000.0
        nextra = int(round(extra_ms / dt))
        t_ms = np.concatenate([t_ms, t_ms[-1] + np.arange(1, nextra+1)*dt])
        for k in list(act_controls.keys()):
            last = act_controls[k][-1] if len(act_controls[k]) else 0.0
            act_controls[k] = np.concatenate([act_controls[k], np.full(nextra, last)])
        print(f"[pulse] You can now add a visible pulse to '{test_act}' before rendering.")
    else:
        print("[pulse] No familiar actuator found to test.")
except Exception as e:
    print("[pulse] Skipped (mujoco import failed):", e)
# ==== end one-cell diagnostics ====


In [ ]:
import mujoco
from pathlib import Path

def list_model_cameras(mjcf_path: Path):
    mjcf_path = Path(mjcf_path)
    m = mujoco.MjModel.from_xml_path(str(mjcf_path))
    print(f"Total cameras: {m.ncam}")
    for i in range(m.ncam):
        name = mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_CAMERA, i)
        # Some parameters live in model fields:
        # pos/orient are in m.cam_pos / m.cam_quat; fovy in m.cam_fovy; mode in m.cam_mode
        pos = m.cam_pos[i]
        fovy = m.cam_fovy[i]
        mode = int(m.cam_mode[i])  # 0: fixed, 1: track, 2: trackcom, 3: targetbody, 4: targetbodycom (depends on MuJoCo version)
        print(f"[#{i:02d}] name='{name}'  mode={mode}  pos=({pos[0]:.3f},{pos[1]:.3f},{pos[2]:.3f})  fovy={fovy:.2f}")

# --- usage ---
list_model_cameras("/path/to/local/flybody-main/flybody/fruitfly/assets/floor.xml")
